# Gaussian Mixture Models

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/probabilistic-models/01-gaussian-mixture-models

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — soft clustering with probabilities

K-means makes **hard** assignments (each point belongs to exactly one cluster). A **Gaussian Mixture
Model** makes **soft** ones: it models the data as a blend of `K` Gaussians and gives each point a
**responsibility** — the probability it came from each component. Fitting uses the **EM algorithm**:
the **E-step** computes responsibilities given the current Gaussians (Bayes' rule), and the **M-step**
re-estimates each Gaussian's mean, covariance, and weight from the responsibility-weighted data. Unlike
k-means, GMMs capture **elliptical** clusters (full covariance) and give calibrated cluster
probabilities. We build 1-D EM from scratch and validate against `sklearn`.

## Soft vs Hard Clustering

GMMs assign **probabilities** (responsibilities) to each cluster, unlike K-Means which makes hard assignments.

In [ ]:
np.random.seed(42)
n = 150
X = np.concatenate([np.random.randn(n) * 0.8 - 2, np.random.randn(n) * 1.2 + 3])

# Simple EM for 1D GMM
K = 2
mu = np.array([-3.0, 2.0])
sigma = np.array([1.0, 1.0])
pi = np.array([0.5, 0.5])

for _ in range(50):
    # E-step
    gamma = np.zeros((len(X), K))
    for k in range(K):
        gamma[:, k] = pi[k] * norm.pdf(X, mu[k], sigma[k])
    gamma /= gamma.sum(axis=1, keepdims=True)
    # M-step
    Nk = gamma.sum(axis=0)
    for k in range(K):
        mu[k] = np.sum(gamma[:, k] * X) / Nk[k]
        sigma[k] = np.sqrt(np.sum(gamma[:, k] * (X - mu[k])**2) / Nk[k])
        pi[k] = Nk[k] / len(X)

# Visualize responsibilities
fig, axes = plt.subplots(2, 1, figsize=(10, 7), gridspec_kw={'height_ratios': [1, 2]})

axes[0].scatter(X, np.zeros_like(X), c=gamma[:, 0], cmap='coolwarm', s=20, alpha=0.8)
axes[0].set_title('Responsibility P(cluster=0) per point', color='white', fontsize=11)
axes[0].set_yticks([])

x_grid = np.linspace(-6, 8, 300)
mixture = sum(pi[k] * norm.pdf(x_grid, mu[k], sigma[k]) for k in range(K))
axes[1].hist(X, bins=50, density=True, color='#818cf8', alpha=0.4, edgecolor='#1a1d27')
axes[1].plot(x_grid, mixture, color='#14b8a6', linewidth=2, label='Mixture')
for k in range(K):
    axes[1].plot(x_grid, pi[k] * norm.pdf(x_grid, mu[k], sigma[k]), '--',
                 color=['#f43f5e', '#eab308'][k], linewidth=1.5,
                 label=f'Component {k+1}: μ={mu[k]:.2f}, σ={sigma[k]:.2f}, π={pi[k]:.2f}')
axes[1].set_title('Gaussian Mixture Fit', color='white', fontsize=11)
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

**What to notice:** after EM converges, each point has a **responsibility** (its color) — points
between the two bumps are genuinely uncertain (purple), not forced into one cluster. The fitted mixture
density (two weighted Gaussians) matches the histogram. This *soft* assignment is GMM's edge over
k-means's hard cut.

## The library way — validate our EM against `sklearn`

`sklearn.mixture.GaussianMixture` runs the same EM. The cell checks our from-scratch means and standard
deviations match it (components are interchangeable, so we sort by mean to align them).

In [ ]:
from sklearn.mixture import GaussianMixture

X_1d = X.copy()   # snapshot: later cells reuse the name X
gm = GaussianMixture(n_components=2, random_state=0).fit(X_1d.reshape(-1, 1))

ours = sorted(zip(mu, sigma))
sk   = sorted(zip(gm.means_.ravel(), np.sqrt(gm.covariances_.ravel())))
print('our  (mean, sd):', [(round(m, 2), round(s, 2)) for m, s in ours])
print('sklearn (mean, sd):', [(round(m, 2), round(s, 2)) for m, s in sk])
assert np.allclose([m for m, _ in ours], [m for m, _ in sk], atol=0.3), "EM means must match sklearn"
print('\nour from-scratch EM == sklearn GaussianMixture ✓')

**What to notice:** our hand-rolled EM recovers the same two Gaussians (`≈ −2` and `≈ 3`) as
`sklearn` — same algorithm, same fit. The E-step/M-step loop is genuinely all there is to it; the
library adds initialization heuristics and covariance-type options.

## Responsibility by hand

The responsibility $\gamma_{ik}=P(z_i{=}k\mid x_i)$ is just Bayes' theorem: prior $\pi_k$ times likelihood $\mathcal N(x_i\mid\mu_k,\sigma_k)$, normalized over components. Below we reproduce the lesson's worked example ($x=2$, two unit Gaussians at $\mu=0,5$) and confirm $\gamma=(0.924, 0.076)$.

In [ ]:
import numpy as np
from scipy.stats import norm

# Worked example: x=2, two components N(0,1) and N(5,1), equal weights
x = 2.0
pi = np.array([0.5, 0.5])
mu = np.array([0.0, 5.0])
sig = np.array([1.0, 1.0])

# Gaussian densities (the 1/sqrt(2pi) cancels in the ratio, kept for clarity)
dens = norm.pdf(x, mu, sig)
print('N(2|0,1) =', round(dens[0], 4), '  N(2|5,1) =', round(dens[1], 5))

# Bayes' theorem -> responsibilities
unnorm = pi * dens
gamma = unnorm / unnorm.sum()
print('responsibilities gamma =', np.round(gamma, 3), ' (lesson: [0.924, 0.076])')

# A point exactly between the means splits 50/50
g_mid = (pi * norm.pdf(2.5, mu, sig)); g_mid /= g_mid.sum()
print('at x=2.5 (midpoint):', np.round(g_mid, 3))

# Sanity: M-step mu is the responsibility-weighted mean (soft K-Means).
# Two points, full E then M on mu, to show the weighting in action.
Xp = np.array([2.0, 2.5])
G = np.array([(pi * norm.pdf(xi, mu, sig)) for xi in Xp])
G /= G.sum(axis=1, keepdims=True)
Nk = G.sum(axis=0)
mu_new = (G * Xp[:, None]).sum(axis=0) / Nk
print('M-step mu (responsibility-weighted mean) =', np.round(mu_new, 3))


**What to notice:** the responsibility-by-hand computation is just **Bayes' rule** — weight each
Gaussian's density by its mixing coefficient `π_k`, then normalize across components. That single
formula *is* the E-step; the M-step is the responsibility-weighted mean/variance.

## 2D GMM with Elliptical Clusters

Unlike K-Means, GMMs can model **elliptical** clusters with different orientations.

In [ ]:
np.random.seed(42)
n = 200

# Create clusters with different shapes
angle = np.pi / 4
R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
X1 = R @ np.diag([3, 0.5]) @ np.random.randn(2, n // 2)          # (2, 100), tilted ellipse at origin
X2 = (np.random.randn(2, n // 2) * 1.5 + np.array([[8], [2]])).T  # (100, 2), blob at (8, 2)
X_2d = np.vstack([X1.T, X2])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# K-Means would force circular clusters
from sklearn.cluster import KMeans
km = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_2d)
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=km.labels_, cmap='viridis', s=10, alpha=0.6)
axes[0].set_title('K-Means (spherical clusters)', color='white', fontsize=11)

# GMM can model ellipses
from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=42).fit(X_2d)
labels = gmm.predict(X_2d)
probs = gmm.predict_proba(X_2d)
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=probs[:, 0], cmap='coolwarm', s=10, alpha=0.7)
axes[1].set_title('GMM (elliptical clusters, soft)', color='white', fontsize=11)

for ax in axes:
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

**What to notice:** on tilted, **elliptical** clusters **k-means fails** (it forces spherical
regions and mislabels the diagonal ellipse), while the **full-covariance GMM** captures the orientation
and shape, with soft probabilities shading the overlap. Modeling covariance is exactly what lets GMMs
handle clusters k-means can't.

## Covariance type and model selection

GMM covariance can be `full`, `tied`, `diag`, or `spherical` — more flexibility means more parameters. **BIC** picks the number of components by penalizing complexity.

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=500, centers=3, cluster_std=1.0, random_state=0)
bics = []
for k in range(1, 7):
    gm = GaussianMixture(n_components=k, covariance_type='full', random_state=0).fit(X)
    bics.append(gm.bic(X))
    print(f'K={k}: BIC = {gm.bic(X):.0f}')
print('best K by BIC:', int(np.argmin(bics)) + 1)

**What to notice:** **BIC** scores each `K` by trading model fit against complexity, and you pick the
`K` that **minimizes** it (printed above). The penalty on extra parameters stops it from always
preferring more components — giving GMMs a principled way to choose `K`, something k-means lacks.

## Gotchas & tradeoffs

- **EM finds local optima.** Different initializations converge to different fits — use multiple restarts
  (`n_init > 1`), as with k-means.
- **Covariance can collapse.** A component can shrink onto a single point, sending its density (and the
  likelihood) to infinity — regularize with a covariance floor (`reg_covar`).
- **You still choose K** (and the covariance type: full/diag/spherical trades flexibility for
  parameters). Use BIC/AIC.
- **Assumes Gaussian components.** If clusters aren't roughly Gaussian, the fit is misleading — GMM is a
  model, not a universal clusterer.

In [ ]:
# EM finds LOCAL optima: on OVERLAPPING clusters, different inits reach different fits
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture
X_overlap, _ = make_blobs(n_samples=400, centers=5, cluster_std=2.8, random_state=1)  # heavy overlap
lls = [GaussianMixture(n_components=5, n_init=1, random_state=s).fit(X_overlap).score(X_overlap)
       for s in range(10)]
print('log-likelihood across 10 single-init runs:', [round(l, 3) for l in lls])
print(f'spread = {max(lls) - min(lls):.3f}  -> EM is init-sensitive; use n_init>1 to keep the best run')

**What to notice:** the same GMM fit from different random seeds reaches **different**
log-likelihoods — EM only guarantees a *local* optimum, not the global one. Running several
initializations and keeping the best (`n_init > 1`) is standard practice, exactly like k-means's
restarts.

## Key takeaways

- A GMM models data as a weighted mix of Gaussians — **soft** assignments (responsibilities).
- Unlike K-Means it captures **elliptical** clusters and gives probabilistic membership.
- **Covariance type** trades flexibility for parameters; `full` is most expressive.
- Choose the number of components with **BIC/AIC**, not the likelihood alone.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Responsibilities

The E-step's core computation: how much does each component **claim** a point?

$$r_k(x) = \frac{\pi_k \, \mathcal{N}(x \mid \mu_k, \sigma_k^2)}{\sum_j \pi_j \, \mathcal{N}(x \mid \mu_j, \sigma_j^2)}$$

Implement it for 1D. The checks pin the intuition: a midpoint between equal components splits 50/50, a point at a component's mean mostly belongs to it, and **priors break ties**.

In [ ]:
def gauss(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))


def responsibilities(x, pis, mus, sigmas):
    """Posterior component probabilities for scalar x. Arrays pis/mus/sigmas line up."""
    pis = np.asarray(pis, dtype=float)
    mus = np.asarray(mus, dtype=float)
    sigmas = np.asarray(sigmas, dtype=float)

    # TODO(you): unnormalized weights pi_k * N(x | mu_k, sigma_k)
    w = ...

    # TODO(you): normalize to sum to 1
    return ...

In [ ]:
# Checks — run me
r = responsibilities(0.0, [0.5, 0.5], [-1.0, 1.0], [1.0, 1.0])
assert abs(r.sum() - 1) < 1e-12, "responsibilities sum to 1"
assert abs(r[0] - 0.5) < 1e-12, "midpoint between equal components -> 50/50"

r = responsibilities(-1.0, [0.5, 0.5], [-1.0, 1.0], [1.0, 1.0])
assert r[0] > 0.8, "a point at component 0's mean mostly belongs to it"

r = responsibilities(0.0, [0.9, 0.1], [-1.0, 1.0], [1.0, 1.0])
assert r[0] > 0.5, "priors break the tie: the 90% component wins at the midpoint"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def responsibilities(x, pis, mus, sigmas):
    pis = np.asarray(pis, dtype=float)
    mus = np.asarray(mus, dtype=float)
    sigmas = np.asarray(sigmas, dtype=float)
    w = pis * gauss(x, mus, sigmas)
    return w / w.sum()
```

</details>

### Exercise 2 — The mixture density

A GMM's density is just the prior-weighted sum of its components:

$$p(x) = \sum_k \pi_k \, \mathcal{N}(x \mid \mu_k, \sigma_k^2)$$

Implement it (vectorized over $x$). Since the weights sum to 1 and each component is normalized, the mixture **still integrates to 1** — the checks verify that numerically.

In [ ]:
def gmm_pdf(x, pis, mus, sigmas):
    """Mixture density evaluated at x (scalar or array)."""
    x = np.atleast_1d(np.asarray(x, dtype=float))
    out = np.zeros_like(x)

    for p, m, s in zip(pis, mus, sigmas):
        # TODO(you): accumulate p * N(x | m, s)
        out += ...

    return out if out.size > 1 else float(out[0])

In [ ]:
# Checks — run me
xs = np.linspace(-10, 10, 4001)
dens = gmm_pdf(xs, [0.3, 0.7], [-2.0, 3.0], [1.0, 0.5])
assert abs(np.trapezoid(dens, xs) - 1) < 1e-6, "a mixture of normalized densities integrates to 1"

assert gmm_pdf(3.0, [0.3, 0.7], [-2.0, 3.0], [1.0, 0.5]) > gmm_pdf(-2.0, [0.3, 0.7], [-2.0, 3.0], [1.0, 0.5]), \
    "the heavier, tighter component has the taller peak"
assert abs(gmm_pdf(0.5, [1.0], [0.5], [2.0]) - gauss(0.5, 0.5, 2.0)) < 1e-12, "one component -> just that Gaussian"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def gmm_pdf(x, pis, mus, sigmas):
    x = np.atleast_1d(np.asarray(x, dtype=float))
    out = np.zeros_like(x)
    for p, m, s in zip(pis, mus, sigmas):
        out += p * gauss(x, m, s)
    return out if out.size > 1 else float(out[0])
```

</details>